# Thermo/FEI Microscope XML Parser

Parse Thermo microscope XML metadata into a typed `CustomData` dictionary and a flattened path/value map for the rest of the XML.

## 1. Set Up Notebook Environment

Import required libraries and define helper functions for parsing the XML.

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET


def _strip_ns(tag):
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


def _parse_typed_value(value_elem):
    if value_elem is None:
        return None
    nil_attr = value_elem.attrib.get("{http://www.w3.org/2001/XMLSchema-instance}nil")
    if nil_attr == "true":
        return None
    raw_text = value_elem.text.strip() if value_elem.text else ""
    type_attr = value_elem.attrib.get("{http://www.w3.org/2001/XMLSchema-instance}type", "")
    type_name = type_attr.split(":", 1)[-1] if ":" in type_attr else type_attr

    if type_name in {"double", "float"}:
        try:
            return float(raw_text)
        except ValueError:
            return raw_text
    if type_name in {"int", "long", "short"}:
        try:
            return int(raw_text)
        except ValueError:
            return raw_text
    if type_name == "boolean":
        return raw_text.lower() == "true"

    return raw_text


def _parse_custom_data(root, namespaces):
    custom_data = {}
    custom_elem = root.find("m:CustomData", namespaces)
    if custom_elem is None:
        return custom_data

    for kv in custom_elem.findall("a:KeyValueOfstringanyType", namespaces):
        key_elem = kv.find("a:Key", namespaces)
        value_elem = kv.find("a:Value", namespaces)
        if key_elem is None:
            continue
        key = key_elem.text.strip() if key_elem.text else ""
        if not key:
            continue
        custom_data[key] = _parse_typed_value(value_elem)

    return custom_data


def _flatten_leaf_text(elem, path, out):
    nil_attr = elem.attrib.get("{http://www.w3.org/2001/XMLSchema-instance}nil")
    if nil_attr == "true":
        out["/".join(path)] = None
        return

    children = list(elem)
    if not children:
        text = elem.text.strip() if elem.text else ""
        if text != "":
            out["/".join(path)] = text
        return

    for child in children:
        _flatten_leaf_text(child, path + [_strip_ns(child.tag)], out)


def parse_microscope_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    namespaces = {
        "m": "http://schemas.datacontract.org/2004/07/Fei.SharedObjects",
        "a": "http://schemas.microsoft.com/2003/10/Serialization/Arrays",
        "i": "http://www.w3.org/2001/XMLSchema-instance",
    }

    data = {
        "customData": _parse_custom_data(root, namespaces),
        "fields": {},
    }

    _flatten_leaf_text(root, [_strip_ns(root.tag)], data["fields"])

    return data

## 2. Load or Create Sample Data

Point to the XML file and parse it.

In [2]:
xml_path = Path("FoilHole_21380846_Data_18931123_17_20251221_055618.xml")

parsed = parse_microscope_xml(xml_path)
parsed.keys()

dict_keys(['customData', 'fields'])

## 3. Run Core Analysis Code

Inspect a few parsed fields and `CustomData` entries.

In [3]:
custom_data = parsed["customData"]
fields = parsed["fields"]

example_keys = [
    "Detectors[BM-Falcon].TotalDose",
    "IlluminationIntensity",
    "AppliedDefocus",
]

{key: custom_data.get(key) for key in example_keys}

{'Detectors[BM-Falcon].TotalDose': 35.3630687638362,
 'IlluminationIntensity': 0.456639999999998,
 'AppliedDefocus': -1.4e-06}

In [6]:
for key in sorted(fields.keys()):
    print(key)

MicroscopeImage/CustomData/KeyValueOfstringanyType/Key
MicroscopeImage/CustomData/KeyValueOfstringanyType/Value
MicroscopeImage/IntensityScale
MicroscopeImage/ReferenceTransformation/matrix/_m11
MicroscopeImage/ReferenceTransformation/matrix/_m12
MicroscopeImage/ReferenceTransformation/matrix/_m21
MicroscopeImage/ReferenceTransformation/matrix/_m22
MicroscopeImage/ReferenceTransformation/matrix/_offsetX
MicroscopeImage/ReferenceTransformation/matrix/_offsetY
MicroscopeImage/ReferenceTransformation/matrix/_padding
MicroscopeImage/ReferenceTransformation/matrix/_type
MicroscopeImage/ReferenceTransformation/unit/_x003C_PrefixExponent_x003E_k__BackingField
MicroscopeImage/ReferenceTransformation/unit/_x003C_Symbol_x003E_k__BackingField
MicroscopeImage/SpatialScale/offset/x/numericValue
MicroscopeImage/SpatialScale/offset/x/unit/_x003C_PrefixExponent_x003E_k__BackingField
MicroscopeImage/SpatialScale/offset/x/unit/_x003C_Symbol_x003E_k__BackingField
MicroscopeImage/SpatialScale/offset/y/num

## 4. Visualize Results

Show the first few flattened fields as a simple table.

In [8]:
import pandas as pd

fields_table = (
    pd.DataFrame(
        sorted(fields.items(), key=lambda item: item[0]),
        columns=["path", "value"],
    )
)
fields_table

# export the data to csv
fields_table.to_csv("microscope_fields.csv", index=False)

## 5. Add Unit Tests for Key Functions

A lightweight sanity check to confirm parsing and flattening.

In [5]:
def _run_tests():
    assert isinstance(parsed, dict)
    assert "customData" in parsed
    assert "fields" in parsed
    assert isinstance(parsed["customData"], dict)
    assert isinstance(parsed["fields"], dict)

    # Spot-check a couple of expected fields
    assert "MicroscopeImage/microscopeData/core/ApplicationSoftware" in parsed["fields"]
    assert parsed["fields"]["MicroscopeImage/microscopeData/core/ApplicationSoftware"] == "EPU"

    print("All tests passed.")


_run_tests()

All tests passed.
